Etiquetado (Tagging)

El Concepto Humano: Imagina que tienes una montaña de cartas sobre tu escritorio. Tu trabajo no es leerlas todas a detalle, sino simplemente ponerles un sello adhesivo arriba que diga en qué idioma están y si el cliente está feliz o enojado. Eso es el etiquetado.

In [10]:
import os
from dotenv import load_dotenv

import os y dotenv: Esto es como sacar tus llaves. Tu API_KEY de Google es secreta. En lugar de escribirla directamente en el código (donde alguien podría verla), la guardas en un archivo cerrado llamado .env. Estas librerías se encargan de abrir ese archivo y sacar la llave por ti de forma segura.

In [11]:

from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

ChatGoogleGenerativeAI: Es el "cerebro" que vamos a usar. LangChain tiene un adaptador especial para conectarse específicamente a los modelos Gemini de Google.

BaseModel y Field (de Pydantic): Pydantic es una librería de Python que nos permite crear "moldes" o "plantillas". Nos asegura que los datos tengan la forma correcta (por ejemplo, que un número sea realmente un número y no una palabra).

In [12]:
# 1. Cargar entorno y modelo Gemini
load_dotenv()

True

load_dotenv(): Abre tu archivo .env para que Python pueda leer la llave.

In [17]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0 # Temperatura 0 para ser estrictos y no inventar
)

llm = ChatGoogleGenerativeAI(...): Aquí estamos "contratando" a nuestro asistente.
model="gemini-2.5-flash-lite"": Le decimos exactamente a quién queremos contratar. Elegimos el modelo "flash" porque es rapidísimo para estas tareas.

api_key=...: Le entregamos la llave secreta para que nos deje usar el servicio.

El detalle clave: temperature=0: La "temperatura" controla la creatividad de la IA. Si es alta (ej. 0.9), la IA será poética y creativa. Como queremos que sea estricta, aburrida y se ciña a las reglas de nuestro etiquetado, la ponemos en 0 (cero creatividad, máxima precisión).

In [18]:
# 2. Definir el "Molde" (El formulario que Gemini debe llenar)
class EtiquetaDocumento(BaseModel):
    """Etiqueta el texto con información específica."""
    sentimiento: str = Field(description="El sentimiento del texto: 'positivo', 'negativo' o 'neutral'")
    idioma: str = Field(description="El idioma del texto (ejemplo: 'es' para español, 'en' para inglés)")

¿Qué es esto? Imagina que estás imprimiendo un formulario en blanco en una imprenta. No quieres que la IA escriba donde quiera. Quieres que llene exactamente las casillas que tú has diseñado.

class EtiquetaDocumento(BaseModel):: Le damos un nombre a nuestro formulario. La palabra BaseModel le dice a Python que esto es un molde estricto de Pydantic.

Las variables (sentimiento e idioma): Estas son las "casillas" del formulario. Le decimos a Python que ambas deben ser cadenas de texto (str).

Field(...) (¡Muy importante!): Aquí es donde ocurre la magia para la IA. La descripción (description="...") es el "manual de instrucciones" que la IA va a leer. Le estás diciendo a Gemini exactamente qué debe buscar para llenar esa casilla.

In [19]:
# 3. La Magia Moderna: Le ponemos el molde al cerebro de Gemini
etiquetador = llm.with_structured_output(EtiquetaDocumento)

Esta es la línea más poderosa y la que reemplaza a decenas de líneas del código antiguo.

Tomamos a nuestro empleado (llm), le entregamos el formulario en blanco (EtiquetaDocumento), y usamos el comando with_structured_output.

A partir de este momento, el objeto etiquetador es una versión de Gemini que tiene "anteojeras". Ya no puede conversar libremente. Su única función en la vida es recibir texto, procesarlo, y devolverte un objeto de Python que coincida perfectamente con la clase EtiquetaDocumento.

In [20]:
# 4. Pruebas de Estudio
resultado_1 = etiquetador.invoke("Me encanta este nuevo sistema, es súper rápido.")
print("Prueba 1:")
print(f"Sentimiento: {resultado_1.sentimiento} | Idioma: {resultado_1.idioma}\n")

resultado_2 = etiquetador.invoke("I really hate how slow the platform is today.")
print("Prueba 2:")
print(f"Sentimiento: {resultado_2.sentimiento} | Idioma: {resultado_2.idioma}")

Prueba 1:
Sentimiento: positivo | Idioma: es

Prueba 2:
Sentimiento: negativo | Idioma: en


invoke(...): Es como entregarle un documento al asistente y decirle: "¡Trabaja!".

Le pasamos la frase: "Me encanta este nuevo sistema, es súper rápido."

Gemini lee la frase, revisa las instrucciones de tu "molde" (Pydantic), y decide cómo llenar las casillas.

El resultado (resultado_1): No te devuelve un texto largo. Te devuelve un objeto de Python limpio y estructurado. Por eso puedes acceder a las variables escribiendo resultado_1.sentimiento (que te dará "positivo") y resultado_1.idioma (que te dará "es").